#Intialize notebook 

In [0]:
from Scripts.Silver_Layer_Scripts.ColumnMapConfig import *
from pyspark.sql import functions as F 
tables = spark.catalog.listTables("data_analytics.bronzelayer")

column_mappings ={
    "customer_location": customer_location,
    "customer_demographic_info" : customer_demographic_info,
    "customer_information" : customer_information,
    "product_catagories": product_catagories,
    "product_information": product_information,
    "sales_details": sales_details
}

silver_layer_tables = spark.catalog.listTables("data_analytics.silverlayer")

In [0]:
%sql
USE CATALOG data_analytics;
USE SCHEMA silverlayer;

In [0]:
from Scripts.Silver_Layer_Scripts.helpers import *
deleteAll("data_analytics", "silverlayer", spark)

#Data transformations

##Table rename
loading tables from the bronze layer to silver layer, at the same time renaming them 

In [0]:
for table in tables:
    (
        spark.read.table(f"{table.catalog}.{table.namespace[0]}.{table.name}")
        .write.mode("overwrite")
        .saveAsTable(table_names[table.name])
    )

##Change column names
mapping names defined in the **ColumnMapConfig.py** script to silver tables

In [0]:
for table in silver_layer_tables:

    spark.sql(f"ALTER TABLE {table.name} SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name')")

    current_columns = spark.read.table(f"{table.catalog}.{table.namespace[0]}.{table.name}").columns
    target_names = column_mappings.get(table.name)
    
    if not target_names:
        print(f"Warning: No mapping found for table {table.name}. Skipping.")
        continue
        
    target_list = list(target_names.values()) if isinstance(target_names, dict) else list(target_names)
    
    for current_name, target_name in zip(current_columns, target_list):
       
        if current_name != target_name:
            rename_query = f"ALTER TABLE {table.name} RENAME COLUMN '{current_name}' TO {target_name};"
            spark.sql(rename_query)
            print(f"  -> Renamed {current_name} to {target_name}")

##TODO Trim whitespaces in string values

In [0]:
for table in silver_layer_tables:
    df = spark.read.table(table.name)
    

##Properly format the date
The sales_details table **('12345678' -> 'yyyy-mm-dd')** <br>
And finally cast the column to the proper datatype

In [0]:
df = spark.read.table("sales_details")

sls_columns = ["Order_date", "Ship_date", "Due_date"]

for column in sls_columns:
    df = df.withColumn(column, F.when(F.length(df[column])  < 8, "19000101").otherwise(df[column]))
    df = df.withColumn(column, F.to_date(df[column], "yyyyMMdd"))
    df = df.withColumn(column, F.date_format(df[column], "yyyy-MM-dd"))
    df = df.withColumn(column, F.expr(f"try_to_date({column})"))


df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("sales_details")

##Normalize the gender values
change the gender values from M -> Male and F -> Female

In [0]:
df = spark.read.table("customer_demographic_info")
df = df.withColumn(
    "Gender",
    F.when(F.upper(df["Gender"]) == "M", "Male")
    .when(F.upper(df["Gender"]) == "F", "Female")
    .otherwise(df["Gender"]),
)
df.write.mode("overwrite").saveAsTable("customer_demographic_info")
###********************READING A DIFFERENT TABLE***********************************
df = spark.read.table("customer_information")
df = (
    df.withColumn(
        "Gender",
        F.when(F.upper(df["Gender"]) == "M", "Male")
        .when(F.upper(df["Gender"]) == "F", "Female")
        .otherwise(df["Gender"]))
    .withColumn(
        "Marital_status",
        F.when(F.upper(df["Marital_status"]) == "M", "Married")
        .when(F.upper(df["Marital_status"]) == "S", "Single")
        .otherwise(df["Marital_status"])
    )
)